# Model Development — Manhattan Urban Heat Prediction

## Purpose

This notebook develops and evaluates predictive models for estimating Universal Thermal Climate Index (UTCI) from environmental conditions.

The notebook implements the modelling workflow defined in Session 4:

1. Load cleaned analysis-ready data
2. Create train/validation/test splits
3. Train a baseline model
4. Train a machine learning model
5. Compare performance
6. Save the selected model
7. Document modelling decisions

The final model will support urban heat-risk assessment and ranking workflows.

In [1]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import joblib

RANDOM_SEED = 42

# Load Datasets

The cleaned dataset was generated during Session 3.

For Session 4, the dataset has already been split into:

- Training set
- Validation set
- Test set

The validation set is used during model selection while the test set remains untouched until final evaluation.

In [2]:
train = pd.read_parquet(
    "../data/processed/manhattan-utci-train.parquet"
)

val = pd.read_parquet(
    "../data/processed/manhattan-utci-val.parquet"
)

test = pd.read_parquet(
    "../data/processed/manhattan-utci-test.parquet"
)

print("train", train.shape)
print("valid", val.shape)
print("test", test.shape)

train (1545, 6)
valid (331, 6)
test (332, 6)


# Feature Selection

The objective is to predict UTCI.

Input variables were selected based on their known influence on outdoor thermal comfort.

Features:

- Air temperature
- Relative humidity
- Wind speed
- Solar radiation

Target:

- UTCI

In [3]:
FEATURES = [
    "air_temp_c",
    "humidity_pct",
    "wind_speed_ms",
    "solar_radiation"
]

TARGET = "utci"

X_train = train[FEATURES]
y_train = train[TARGET]

X_val = val[FEATURES]
y_val = val[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

# Baseline Model

A baseline establishes the minimum acceptable performance level.

The baseline predicts the average UTCI value observed in the training data.

Any useful machine learning model should outperform this benchmark.

In [4]:
baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_val)

baseline_mae = mean_absolute_error(
    y_val,
    baseline_predictions
)

baseline_r2 = r2_score(
    y_val,
    baseline_predictions
)

print("BASELINE")
print("MAE:", baseline_mae)
print("R2 :", baseline_r2)

BASELINE
MAE: 3.86173978040458
R2 : -1.1325911034193399


# Random Forest Model

Random Forest was selected because:

- It handles nonlinear relationships
- It requires minimal feature engineering
- It is robust to noisy environmental data
- It provides feature importance estimates

The model is trained on the training set and evaluated on the validation set.

In [5]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_SEED
)

rf.fit(X_train, y_train)

rf_predictions = rf.predict(X_val)

rf_mae = mean_absolute_error(
    y_val,
    rf_predictions
)

rf_r2 = r2_score(
    y_val,
    rf_predictions
)

print("RANDOM FOREST")
print("MAE:", rf_mae)
print("R2 :", rf_r2)

RANDOM FOREST
MAE: 0.21851080060423234
R2 : 0.9918801265369813


# Model Comparison

Performance is evaluated using:

## Mean Absolute Error (MAE)

Measures average prediction error.

Lower values are better.

## R²

Measures proportion of variance explained.

Higher values are better.

A successful model should reduce MAE and increase R² relative to the baseline.

In [6]:
results = pd.DataFrame({
    "Model": ["Baseline", "Random Forest"],
    "MAE": [baseline_mae, rf_mae],
    "R2": [baseline_r2, rf_r2]
})

results

,Model,MAE,R2
0,Baseline,3.861740,-1.132591
1,Random Forest,0.218511,0.991880


# Feature Importance

Random Forest provides an estimate of the relative contribution of each feature.

This helps interpret the model and identify the strongest drivers of thermal comfort conditions.

In [7]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": rf.feature_importances_
})

importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
0,air_temp_c,0.869367
2,wind_speed_ms,0.114509
3,solar_radiation,0.013046
1,humidity_pct,0.003079


# Save Model

The selected model is persisted so it can be reused by downstream workflows without retraining.

In [8]:
joblib.dump(
    rf,
    "../models/baseline.joblib"
)

print("Model saved.")

Model saved.


# Final Evaluation Summary

## Baseline

- MAE ≈ 3.86
- R² ≈ -1.13

## Random Forest

- MAE ≈ 0.22
- R² ≈ 0.99

## Decision

The Random Forest substantially outperforms the baseline and is selected as the Session 4 production model.

## Limitations

- Trained on a relatively small environmental dataset
- Represents Manhattan conditions only
- Does not account for future climate scenarios
- Not intended for operational forecasting

## Next Steps

Session 5 will investigate synthetic data generation and robustness testing.